# HW04 LSTM Helper

Use this notebook for **Part II of HW4**.  
Replace the data-loading section with your own dataset, then run through to produce the three required outputs:
1. Training curves
2. Predictions on train / validation / test sets
3. SHAP memory depth

Refer to `02_LSTM_dynamic_inputs.ipynb` for detailed explanations of every step.

In [1]:
# Google Colab setup (skip if running locally)
!pip install -q numpy pandas matplotlib torch shap

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import shap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


---
## Part 1 — Load Your Data

**Q7**: Describe your inputs and outputs, and how you split your data into train / val / test.

Your CSV should have one row per day with columns for:
- `date` — in any standard format (YYYY-MM-DD)
- forcing variables (e.g. precipitation, temperature, …)
- target variable (e.g. discharge, groundwater level, …)

In [ ]:
# ── TODO: replace with your own file path and column names ───────────────────
DATA_PATH    = 'your_data.csv'          # path to your CSV
FORCING_COLS = ['prcp', 'tmax', 'tmin'] # forcing / input columns
TARGET_COL   = 'discharge'              # output column
DATE_COL     = 'date'                   # date column

TRAIN_END    = '2010-12-31'             # last day of training period
VAL_END      = '2014-12-31'            # last day of validation period
# test period = everything after VAL_END
# ─────────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL]).sort_values(DATE_COL)
df = df.dropna(subset=FORCING_COLS + [TARGET_COL]).reset_index(drop=True)

dates      = df[DATE_COL].values
train_mask = dates <= np.datetime64(TRAIN_END)
val_mask   = (dates > np.datetime64(TRAIN_END)) & (dates <= np.datetime64(VAL_END))
test_mask  = dates > np.datetime64(VAL_END)

print(f'Total days : {len(df)}')
print(f'Train      : {train_mask.sum()}  |  Val: {val_mask.sum()}  |  Test: {test_mask.sum()}')
print(f'Forcing variables ({len(FORCING_COLS)}): {FORCING_COLS}')
print(f'Target: {TARGET_COL}')

In [ ]:
# Normalise forcings and target using training-period statistics
F = df[FORCING_COLS].values.astype(np.float32)
F_mu  = F[train_mask].mean(axis=0)
F_std = F[train_mask].std(axis=0) + 1e-6
F_norm = (F - F_mu) / F_std

y     = df[TARGET_COL].values.astype(np.float32)
y_mu  = y[train_mask].mean()
y_std = y[train_mask].std() + 1e-6
y_norm = (y - y_mu) / y_std

print(f'Forcing shape : {F_norm.shape}  (days × features)')
print(f'Target mean (train): {y_mu:.4f}  std: {y_std:.4f}')

---
## Part 2 — Sliding Window Dataset

In [ ]:
SEQ_LEN = 365  # look-back window in days

class TimeSeriesDataset(Dataset):
    def __init__(self, F_norm, y_norm, seq_len, mask):
        self.samples = []
        for t in range(seq_len - 1, len(y_norm)):
            if mask[t]:
                x = F_norm[t - seq_len + 1 : t + 1]   # (seq_len, n_features)
                self.samples.append((x, y_norm[t]))

    def __len__(self):  return len(self.samples)
    def __getitem__(self, i):
        x, y = self.samples[i]
        return torch.tensor(x), torch.tensor([[y]])

train_ds = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, train_mask)
val_ds   = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, val_mask)
test_ds  = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, test_mask)

print(f'Train sequences: {len(train_ds)}  |  Val: {len(val_ds)}  |  Test: {len(test_ds)}')

---
## Part 3 — LSTM Model

In [ ]:
# ── TODO: adjust hidden_size and num_layers if needed ────────────────────────
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
# ─────────────────────────────────────────────────────────────────────────────

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.2 if num_layers > 1 else 0.0)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMModel(len(FORCING_COLS), HIDDEN_SIZE, NUM_LAYERS).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

---
## Part 4 — Train

In [ ]:
# ── TODO: adjust if needed ────────────────────────────────────────────────────
EPOCHS     = 20
LR         = 1e-3
BATCH_SIZE = 64
# ─────────────────────────────────────────────────────────────────────────────

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

train_losses, val_losses = [], []
best_val, best_state = np.inf, None

for epoch in range(EPOCHS):
    model.train()
    bl = []
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        bl.append(loss.item())
    train_losses.append(np.mean(bl))

    model.eval()
    with torch.no_grad():
        vl = np.mean([criterion(model(X.to(device)), y.to(device)).item()
                      for X, y in val_loader])
    val_losses.append(vl)
    if vl < best_val:
        best_val = vl
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch+1:3d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}')

model.load_state_dict(best_state)

---
## Part 5 — Required Plots
### Q8 — include all three plots in your submission

In [ ]:
# Plot 1: Training curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, color='steelblue', lw=1.5, label='Train')
ax.plot(val_losses,   color='tomato',    lw=1.5, ls='--', label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalised)')
ax.set_title('Training Curves'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Plot 2: Predictions on train / val / test
def predict_period(mask):
    ds  = TimeSeriesDataset(F_norm, y_norm, SEQ_LEN, mask)
    ldr = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0)
    model.eval()
    preds = []
    with torch.no_grad():
        for X, _ in ldr:
            preds.append(model(X.to(device)).cpu().numpy())
    y_hat = np.concatenate(preds).flatten() * y_std + y_mu
    y_obs = y_norm[mask][SEQ_LEN - 1:] * y_std + y_mu
    t_idx = np.where(mask)[0][SEQ_LEN - 1:]
    return dates[t_idx], y_obs, y_hat

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
for ax, mask, label, color in zip(
        axes,
        [train_mask, val_mask, test_mask],
        ['Train', 'Validation', 'Test'],
        ['steelblue', 'orange', 'seagreen']):
    t, obs, sim = predict_period(mask)
    nse = 1 - np.sum((obs - sim)**2) / np.sum((obs - obs.mean())**2)
    ax.plot(t, obs, color='k',     lw=0.8, label='Observed')
    ax.plot(t, sim, color=color,   lw=0.8, label=f'Simulated  NSE={nse:.3f}')
    ax.set_ylabel(TARGET_COL); ax.set_title(label); ax.legend(fontsize=8)
plt.suptitle('LSTM Predictions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Plot 3: SHAP memory depth
np.random.seed(42)
BG_SIZE   = 50
N_EXPLAIN = 50

bg_idx  = np.random.choice(len(train_ds), BG_SIZE,   replace=False)
te_idx  = np.random.choice(len(test_ds),  N_EXPLAIN, replace=False)

X_bg = torch.stack([train_ds[i][0] for i in bg_idx]).to(device)
X_te = torch.stack([test_ds[i][0]  for i in te_idx]).to(device)

model.eval()
explainer = shap.GradientExplainer(model, X_bg)
shap_raw  = explainer.shap_values(X_te)

sv = np.squeeze(shap_raw)                          # (N, SEQ_LEN, n_features)
if sv.ndim == 3 and sv.shape[1] == len(FORCING_COLS):
    sv = sv.transpose(0, 2, 1)

mean_over_time = np.abs(sv).mean(axis=(0, 2))      # (SEQ_LEN,)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(SEQ_LEN), mean_over_time, color='darkorange', lw=1.5)
ax.set_xlabel('Timestep (day 0 = oldest in window)')
ax.set_ylabel('Mean |SHAP|')
ax.set_title('SHAP Memory Depth — How far back does the LSTM look?')
plt.tight_layout(); plt.show()